# Notebook 10 — Expansion–Growth Consistency (Boss #6)

## CONTRACT (frozen; see `canon/20-cosmology/08-notebook10-prereg.md`, commit `54f9717`, and `09-sign-derivation.md`)

**Prediction:** one shape s(z;γ)=(1+z)^(−γ) fits both sides: w(z) = −1 + A_w·s(z;γ_E) (expansion) and μ(z) = 1 + A_μ·s(z;γ_G) (growth), with **γ_E = γ_G, A_w > 0, A_μ > 0** (signs committed pre-data — the growth sign runs against the S8 trend).

**Baseline:** flat ΛCDM (A_w = A_μ = 0) and independent-shapes fit (γ_E free of γ_G).

**Data (pinned at step 4/5; loader refuses to run until files + sha256 exist in `data/cosmo/CHECKSUMS.txt`):**
- Expansion: DESI DR2 BAO distance table (published values + covariance, arXiv ID pinned on ingest) · Pantheon+ SNe binned distance moduli · Planck-2018 distance priors (R, l_a, ω_b).
- Growth: a published fσ₈(z) compilation pinned by citation count at ingest, NOT by result · Planck lensing amplitude as cross-check only.

**Pass criterion:** expansion side prefers A_w ≠ 0 at ≥ 2σ AND |γ_E − γ_G| ≤ 2σ_comb AND A_μ > 0 → UNIFICATION SURVIVES ITS FIRST TEST.

**Falsifiers (frozen verdict tree):** shape mismatch > 2σ, or growth demands A_μ ≤ 0 while expansion demands A_w > 0 at ≥ 3σ → **UNIFICATION DEAD** (boss #6 falls against us; P07 cross-scale claim falls with it). Both sides A ≈ 0 → INDISTINGUISHABLE, no claim. Internally inconsistent growth data (> 3σ) → DATA NOT READY, parked.

**Execution order discipline:** expansion fit (step 4) commits before any growth file is opened (step 5). `FIT_BEFORE_FREEZE` chain-by-chain. `POSTDICTED_SIGN` guard: signs may only come from `09-sign-derivation.md`.

In [ ]:
# Model layer — pure functions, no data. Deterministic; no RNG anywhere.
import numpy as np
from scipy.integrate import quad, solve_ivp

def s_shape(z, gamma):
    return (1.0 + z) ** (-gamma)

def w_of_z(z, A_w, gamma_E):
    return -1.0 + A_w * s_shape(z, gamma_E)

def rho_de_ratio(z, A_w, gamma_E):
    """rho_DE(z)/rho_DE(0) = exp(3 * integral_0^z (1+w)/(1+z') dz').
    With 1+w = A_w (1+z)^(-g): 3*A_w/g * (1 - (1+z)^(-g)) for g != 0."""
    g = gamma_E
    if abs(g) < 1e-12:
        return np.exp(3.0 * A_w * np.log1p(z))
    return np.exp(3.0 * A_w / g * (1.0 - (1.0 + z) ** (-g)))

def E_of_z(z, Om, A_w, gamma_E):
    """H(z)/H0, flat universe (radiation added at ingest if priors need it)."""
    return np.sqrt(Om * (1 + z) ** 3 + (1 - Om) * rho_de_ratio(z, A_w, gamma_E))

def comoving_distance(z, Om, A_w, gamma_E, H0=70.0, c=299792.458):
    integrand = lambda zz: 1.0 / E_of_z(zz, Om, A_w, gamma_E)
    val, _ = quad(integrand, 0.0, z, limit=200)
    return c / H0 * val

def growth_fsigma8(z_out, Om, A_w, gamma_E, A_mu, gamma_G, sigma8_0):
    """Solve d2D/dlna2 + (2 + dlnH/dlna) dD/dlna = 1.5 * mu(z) * Om(a) * D,
    with mu = 1 + A_mu * s(z; gamma_G). Background feels (A_w, gamma_E) only."""
    def rhs(lna, y):
        a = np.exp(lna); z = 1.0 / a - 1.0
        E = E_of_z(z, Om, A_w, gamma_E)
        dlnE = (np.log(E_of_z(z - 1e-5 if z > 1e-4 else 0.0, Om, A_w, gamma_E))
                - np.log(E_of_z(z + 1e-5, Om, A_w, gamma_E))) / (2e-5) * (-(1 + z))
        Om_a = Om * (1 + z) ** 3 / E ** 2
        mu = 1.0 + A_mu * s_shape(z, gamma_G)
        D, Dp = y
        return [Dp, -(2.0 + dlnE) * Dp + 1.5 * mu * Om_a * D]
    lna0, lna1 = np.log(1 / (1 + 30.0)), 0.0
    sol = solve_ivp(rhs, (lna0, lna1), [np.exp(lna0), np.exp(lna0)], dense_output=True,
                    rtol=1e-8, atol=1e-10)
    lna_out = np.log(1.0 / (1.0 + np.atleast_1d(z_out)))
    D = sol.sol(lna_out)[0] / sol.sol(0.0)[0]
    Dp = sol.sol(lna_out)[1] / sol.sol(0.0)[0]
    f = Dp / D
    return f * sigma8_0 * D

In [ ]:
# Data guard — step 4/5 may only proceed once pinned files + checksums exist.
import os, hashlib, sys
DATA = os.path.join('..', 'data', 'cosmo')
REQUIRED_EXPANSION = ['desi_dr2_bao.csv', 'pantheon_plus_binned.csv', 'planck2018_priors.csv']
REQUIRED_GROWTH = ['fsigma8_compilation.csv']

def guard(files):
    ck = os.path.join(DATA, 'CHECKSUMS.txt')
    if not os.path.exists(ck):
        raise RuntimeError('DATA NOT PINNED: data/cosmo/CHECKSUMS.txt missing — '
                           'ingest step (with arXiv IDs recorded) has not run. '
                           'The CONTRACT forbids proceeding.')
    pinned = dict(line.split()[::-1] for line in open(ck) if line.strip())
    for f in files:
        p = os.path.join(DATA, f)
        assert os.path.exists(p), f'missing {f}'
        h = hashlib.sha256(open(p, 'rb').read()).hexdigest()
        assert pinned.get(f) == h, f'checksum mismatch for {f}'
    return True

print('Skeleton OK. Expansion fit blocked until guard(REQUIRED_EXPANSION) passes;')
print('growth files may not even be DOWNLOADED until the expansion fit is committed.')